# 02. 데이터 분석 — XGBoost + SHAP

## 이 노트북이 하는 일
1. XGBoost로 LEED 등급 예측 (5-Fold CV)
2. SHAP으로 등급 결정 요인 분석
3. Robustness 검증 (subset별 비교)
4. 결과 표/그림을 `results/` 에 저장

## 주요 결과
- **EA(에너지)가 Top SHAP feature** (어느 subset이든)
- CV 정확도 ~0.82, Weighted F1 ~0.82

## 참고
- `docs/02_파이프라인_및_분석.md`

## Setup

In [ ]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent
os.chdir(ROOT)
sys.path.insert(0, str(NOTEBOOK_DIR))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder

TABLE_DIR = ROOT / 'results' / 'tables'
FIG_DIR = ROOT / 'results' / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

GRADE_ORDER = ['Certified', 'Silver', 'Gold', 'Platinum']
FEATURE_COLS = ['ratio_LT', 'ratio_SS', 'ratio_WE', 'ratio_EA',
                'ratio_MR', 'ratio_EQ', 'ratio_IP']

## 1단계: 데이터 로드

Option A parquet이 있으면 사용, 없으면 기본 parquet.

In [ ]:
option_a_path = ROOT / 'data' / 'project_features_option_a.parquet'
default_path = ROOT / 'data' / 'project_features.parquet'
PARQUET = option_a_path if option_a_path.exists() else default_path
df = pd.read_parquet(PARQUET)
print(f'사용 데이터: {PARQUET.name}')
print(f'행수: {len(df)}')
print(f"등급: {df['certification_level'].value_counts().to_dict()}")
print(f"버전: {df['original_version'].value_counts().sort_index().to_dict()}")
if 'has_llm_review' in df.columns:
    print(f"LLM 리뷰됨: {df['has_llm_review'].sum()}건")

## 2단계: 공통 함수

In [ ]:
def prepare_xy(data):
    ver_map = {'v2.0': 0, 'v2.2': 1, 'v2009': 2, 'v4': 3, 'v4.1': 4}
    data = data.copy()
    data['version_ord'] = data['original_version'].map(ver_map)
    data['log_area'] = np.log1p(data['gross_area_sqm'].fillna(0))
    cols = FEATURE_COLS + ['log_area', 'version_ord']
    X = data[cols].fillna(0)
    le = LabelEncoder()
    le.fit(GRADE_ORDER)
    y = le.transform(data['certification_level'])
    return X, y

def train_cv(X, y):
    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, eval_metric='mlogloss', verbosity=0,
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    acc = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    f1 = cross_val_score(model, X, y, cv=cv, scoring='f1_weighted')
    model.fit(X, y)
    return model, acc, f1

def compute_shap_top(model, X):
    expl = shap.TreeExplainer(model)
    sv = expl.shap_values(X)
    if isinstance(sv, np.ndarray) and sv.ndim == 3:
        sv = [sv[:, :, c] for c in range(sv.shape[2])]
    mean_abs = np.abs(np.stack(sv, axis=0)).mean(axis=(0, 1))
    return sv, pd.Series(mean_abs, index=X.columns).sort_values(ascending=False)

## 3단계: 전체 460건 기본 분석

In [ ]:
mask_full = df['certification_level'].notna() & (df['certification_level'] != '')
X, y = prepare_xy(df[mask_full])
model, cv_acc, cv_f1 = train_cv(X, y)
sv, top = compute_shap_top(model, X)

print(f'N = {len(X)}')
print(f'CV 정확도: {cv_acc.mean():.4f} ± {cv_acc.std():.4f}')
print(f'CV F1:     {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print(f'\nSHAP Top 5:')
print(top.head(5).round(4))

## 4단계: 주요 figure 생성

In [ ]:
# Figure 1: SHAP bar plot
fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
top_ordered = top[::-1]
ax.barh(top_ordered.index, top_ordered.values, color='#1976D2')
ax.set_xlabel('Mean |SHAP|')
ax.set_title('SHAP Feature Importance (N=460)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'Figure1_shap_importance.png')
plt.close()

# Figure 2: 등급별 카테고리 달성률
grade_mean = df[mask_full].groupby('certification_level')[FEATURE_COLS].mean()
grade_mean = grade_mean.reindex(GRADE_ORDER)
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
grade_mean.plot(kind='bar', ax=ax)
ax.set_ylabel('평균 달성률')
ax.set_title('등급별 카테고리 달성률 비교')
ax.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / 'Figure2_grade_factors.png')
plt.close()

print(f'Figures 저장: {FIG_DIR}')

## 5단계: Robustness 검증

여러 subset에서 Top SHAP feature가 일관되게 나오는지.

In [ ]:
subsets = {
    '전체': mask_full,
    'credit_hit>0.7': mask_full & (df['credit_rule_hit_rate'].fillna(0) > 0.7),
    '구버전': mask_full & df['original_version'].isin(['v2.0', 'v2.2', 'v2009']),
    '신버전': mask_full & df['original_version'].isin(['v4', 'v4.1']),
}
if 'has_llm_review' in df.columns:
    subsets['LLM 리뷰됨'] = mask_full & (df['has_llm_review'] == True)

results = []
for label, mask in subsets.items():
    sub = df[mask]
    if len(sub) < 20:
        continue
    X_s, y_s = prepare_xy(sub)
    m, acc, f1 = train_cv(X_s, y_s)
    _, top_s = compute_shap_top(m, X_s)
    results.append({
        'subset': label, 'N': len(X_s),
        'CV_Acc': round(acc.mean(), 4),
        'CV_F1': round(f1.mean(), 4),
        'Top1': top_s.index[0], 'Top2': top_s.index[1], 'Top3': top_s.index[2],
    })

robust_df = pd.DataFrame(results)
print(robust_df.to_string(index=False))

## 6단계: 표 저장

In [ ]:
t1 = pd.DataFrame({
    '항목': ['총 건물', '등급: Gold', '등급: Silver', '등급: Platinum', '등급: Certified',
             'v4', 'v2009', 'v4.1', 'v2.2', 'v2.0'],
    '건수': [
        len(df),
        (df['certification_level'] == 'Gold').sum(),
        (df['certification_level'] == 'Silver').sum(),
        (df['certification_level'] == 'Platinum').sum(),
        (df['certification_level'] == 'Certified').sum(),
        (df['original_version'] == 'v4').sum(),
        (df['original_version'] == 'v2009').sum(),
        (df['original_version'] == 'v4.1').sum(),
        (df['original_version'] == 'v2.2').sum(),
        (df['original_version'] == 'v2.0').sum(),
    ]
})
t1.to_csv(TABLE_DIR / 'Table1_dataset.csv', index=False, encoding='utf-8-sig')

t2 = pd.DataFrame([{
    'Model': 'XGBoost',
    'CV_Accuracy_mean': round(cv_acc.mean(), 4),
    'CV_Accuracy_std': round(cv_acc.std(), 4),
    'CV_F1_mean': round(cv_f1.mean(), 4),
    'CV_F1_std': round(cv_f1.std(), 4),
    'n_samples': len(X),
    'n_features': X.shape[1],
}])
t2.to_csv(TABLE_DIR / 'Table2_performance.csv', index=False, encoding='utf-8-sig')

t3 = pd.DataFrame({
    'rank': range(1, len(top) + 1),
    'feature': top.index,
    'mean_abs_shap': top.values.round(4),
})
t3.to_csv(TABLE_DIR / 'Table3_shap_top.csv', index=False, encoding='utf-8-sig')

robust_df.to_csv(TABLE_DIR / 'Table4_robustness.csv', index=False, encoding='utf-8-sig')

print(f'Tables 저장: {TABLE_DIR}')

## 결과 요약

In [ ]:
print('=' * 60)
print('  LEEDGRAPH 분석 결과 요약')
print('=' * 60)
print(f'\n[모델 성능] N={len(X)}')
print(f'  CV 정확도: {cv_acc.mean():.4f} ± {cv_acc.std():.4f}')
print(f'  CV F1:     {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print(f'\n[SHAP Top 3]')
for i, (feat, val) in enumerate(top.head(3).items(), 1):
    print(f'  {i}. {feat:15s}  {val:.4f}')
print(f'\n[Robustness] 모든 subset에서 Top1 = {robust_df["Top1"].iloc[0]}')
print(f'  {"✅ 일관" if robust_df["Top1"].nunique() == 1 else "⚠️ 불일치"}')